## Settings

In [1]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [2]:
## libraries
import sys
import numpy as np
import pandas as pd
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import logo_cross_valid
from src.evaluators.metrics import consensus_metrics
from src.evaluators.metrics import CONSENSUS_METRICS
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)
from src.vectorizers.scalers import _log_transformer

## Initialization

In [3]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

Original Data: 32 features, 25 observations


## Training

In [4]:
## leave-one-group-out prediction-quality evaluation
if "results_list" not in globals():
    results_list = list()
    y_true_full = _log_transformer(data[TARGET]).astype(float).to_numpy()
    groups = data["domain"].to_numpy()

    for name, model in models.items():
        print(f"Training {name}...")
        _, y_pred = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "domain",
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )

        for group_name in sorted(pd.Series(groups).dropna().unique()):
            valid = (
                (groups == group_name)
                & np.isfinite(y_true_full)
                & np.isfinite(y_pred)
            )
            if int(np.sum(valid)) < 2:
                continue

            metrics = consensus_metrics(
                y_true = y_true_full[valid],
                y_pred = y_pred[valid]
            )
            results_list.append({
                "model": name,
                "group": group_name,
                **metrics
            })

Training linear_quantile...
Training linear_convex...
Training linear_laws...
Training forest_quantile...
Training boosted_quantile...
Training xgboost_quantile...
Training neural_quantile...
Training neural_expectile...
Training neural_convex...


## Post-Processing

In [5]:
## convert to dataframe
results_data = pd.DataFrame(results_list)
results_data = results_data[["model", "group", *CONSENSUS_METRICS]]
results_data = results_data.sort_values(by = ["model", "group"]).reset_index(drop = True)

## Prediction Quality
This section tests whether held-out predictions recover observed capacity structure across (model, domain) folds. Each learner is trained under leave-one-group-out cross validation by domain. For every held-out domain, structural agreement is computed between the fold-local prediction vector and the observed target. Prediction quality is evaluated via structural agreement rather than residual norms because N3S separability concerns preservation of rank structure and dependence structure rather than absolute-scale recovery, and frontier estimation remains scale-arbitrary across fitting methods.

### Reporting Convention
The table reports medians and quartiles across all 45 (model, domain) folds. `rho` measures global rank agreement between predictions and the observed target. `rbo` emphasizes agreement near the top of the observed capacity ranking. `dcr` captures nonlinear dependence. `ci` is the geometric mean composite over `rho`, `rbo`, and `dcr`. Higher values indicate stronger structural recovery of observed capacities.

In [8]:
## summarize prediction-quality metrics
results_summary = pd.DataFrame(
    {
        metric: [
            results_data[metric].median(),
            results_data[metric].quantile(0.25),
            results_data[metric].quantile(0.75)
        ]
        for metric in CONSENSUS_METRICS
    },
    index = ["Median", "Q1", "Q3"]
).round(3)

display(results_summary)

,rho,rbo,dcr,ci
Median,0.7,0.910,0.836,0.828
Q1,0.3,0.810,0.704,0.523
Q3,0.9,0.982,0.970,0.929


Prediction quality is strong across held-out domains. Median Spearman rho is 0.700. Median RBO is 0.910. Median distance correlation is 0.836. Median CI is 0.828. Fitted frontiers therefore recover the observed capacity structure on unseen domains, establishing prediction quality as the third foundational N3S pillar before downstream falsification and perturbation analyses stress-test its failure modes and robustness.